# DNAmFitAgeGrip

Reproducible conversion of the published DNAmFitAge female and male grip strength regressions into one sex-gated pyaging artifact. The model keeps each coefficient table and reference-median vector independent; the public `female` input selects the corresponding branch.


## Imports and model


In [1]:
import hashlib
import math
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import torch

import pyaging as pya

model = pya.models.DNAmFitAgeGrip()


## Curated clock metadata


In [2]:
# ruff: noqa: E501
model.metadata["clock_name"] = "dnamfitagegrip"
model.metadata["data_type"] = "DNA methylation"  # Paper: Blood DNA methylation was used to develop the fitness biomarkers.
model.metadata["species"] = "Homo sapiens"  # Paper: The development cohorts were human adult studies (FHS, BLSA, and Budapest).
model.metadata["year"] = 2023
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "McGreevy, K. M., et al. “DNAmFitAge: biological age indicator incorporating physical fitness.” Aging 15(10): 3904–3938 (2023)."
model.metadata["doi"] = "https://doi.org/10.18632/aging.204538"
model.metadata["notes"] = "Sex-gated blood DNAm maximum-handgrip-strength estimator using the published female and male regressions selected by the female input."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: The biomarkers were developed from blood DNAm data.
model.metadata["predicts"] = ["grip strength"]  # Paper: The algorithms generate DNAmGaitspeed, DNAmGripmax, or DNAmVO2max estimates in the corresponding physical-fitness scale.
model.metadata["training_target"] = ["grip strength"]  # Paper: The directly measured fitness parameter was the dependent variable in LASSO regression.
model.metadata["unit"] = ["kilograms"]  # Paper: Gait speed is measured in m/s, grip force in kg, and VO2max in mL/kg/min.
model.metadata["model_type"] = "LASSO regression"  # Paper: Each fitness DNAm biomarker was developed using LASSO penalized regression with ten-fold cross-validation.
model.metadata["platform"] = ["Illumina 450K"]  # Paper: The reported fitness CpG background and fitted loci were on the 450K array.
model.metadata["population"] = "adults"  # Paper: The female and male models were fit separately in adult development cohorts.
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 183
model.metadata["citations"] = 99
model.metadata["citations_date"] = "2026-07-05"


## Download and export the published model source

The authors' repository is cloned into this notebook's temporary working directory. R reads the RDS directly and exports only the two coefficient tables and sex-specific median vectors required here.


In [3]:
github_url = "https://github.com/kristenmcgreevy/DNAmFitAge.git"
upstream_commit = "5ba04b4b0ea3b25551c85ae93d8de11903a75fbc"
rds_sha256 = "ebcd6a11e2f6c1088cff4964e04005bd10a33e49871232c9b27ff880e3a73388"
repository = Path("DNAmFitAge")
if repository.exists():
    shutil.rmtree(repository)
subprocess.run(["git", "clone", "--no-checkout", github_url, str(repository)], check=True)
subprocess.run(["git", "-C", str(repository), "checkout", "--detach", upstream_commit], check=True)
resolved_commit = subprocess.run(
    ["git", "-C", str(repository), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert resolved_commit == upstream_commit
rds_path = repository / "DNAmFitnessModelsandFitAge_Oct2022.rds"
hasher = hashlib.sha256()
with rds_path.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        hasher.update(chunk)
assert hasher.hexdigest() == rds_sha256

r_script = """
DNAmFitnessModels <- readRDS("DNAmFitAge/DNAmFitnessModelsandFitAge_Oct2022.rds")
write.csv(DNAmFitnessModels$Grip_noAge_Females, "Grip_noAge_Females.csv")
write.csv(DNAmFitnessModels$Grip_noAge_Males, "Grip_noAge_Males.csv")
write.csv(DNAmFitnessModels$Female_Medians_All, "FemaleMedians.csv")
write.csv(DNAmFitnessModels$Male_Medians_All, "MaleMedians.csv")
"""
r_script_path = Path("download_dnamfitagegrip.r")
r_script_path.write_text(r_script)
subprocess.run(["Rscript", str(r_script_path)], check=True)


Cloning into 'DNAmFitAge'...


HEAD is now at 5ba04b4 Update README.md


CompletedProcess(args=['Rscript', 'download_dnamfitagegrip.r'], returncode=0)

## Build the gated model

Feature order is the first-occurrence ordered union of the published female and male feature sequences, followed by `female`. Methylation entries in the public reference vector are NaN sentinels, so each branch applies its own medians during the forward pass.


In [4]:
# ruff: noqa: E501
def ordered_union(*groups):
    return list(dict.fromkeys(feature for group in groups for feature in group))


def load_component(table_name, medians_name):
    table = pd.read_csv(table_name, index_col=0)
    features = table["term"].iloc[1:].tolist()
    coefficients = torch.tensor(table["estimate"].iloc[1:].tolist()).unsqueeze(0)
    intercept = torch.tensor([table["estimate"].iloc[0]])
    component = pya.models.LinearModel(input_dim=len(features))
    component.linear.weight.data = coefficients.float()
    component.linear.bias.data = intercept.float()
    medians = pd.read_csv(medians_name, index_col=0)
    reference_values = medians.loc[1, features].tolist()
    return component, features, reference_values

female_model, female_features, female_references = load_component("Grip_noAge_Females.csv", "FemaleMedians.csv")
male_model, male_features, male_references = load_component("Grip_noAge_Males.csv", "MaleMedians.csv")

model.features = ordered_union(female_features, male_features) + ["female"]
assert len(model.features) == 183
assert "cg16736630" in male_features
feature_indices = {feature: index for index, feature in enumerate(model.features)}
model.female_model = female_model
model.male_model = male_model
model.female_feature_indices = torch.tensor([feature_indices[feature] for feature in female_features])
model.male_feature_indices = torch.tensor([feature_indices[feature] for feature in male_features])
model.female_reference_values = female_references
model.male_reference_values = male_references
model.female_index = feature_indices["female"]
model.reference_values = [float("nan")] * (len(model.features) - 1) + [1.0]
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]

assert len(model.female_reference_values) == len(female_features)
assert len(model.male_reference_values) == len(male_features)
assert len(model.feature_units) == 183
assert model.reference_values[-1] == 1.0
print({"public_features": len(model.features), "female_features": len(female_features), "male_features": len(male_features)})


{'public_features': 183, 'female_features': 91, 'male_features': 93}


## Verify and save


In [5]:
pya.utils.print_model_details(model)

input_values = torch.full((2, len(model.features)), 0.5, dtype=torch.float64)
input_values[:, model.female_index] = torch.tensor([0.0, 1.0])
model.to(torch.float64).eval()
with torch.no_grad():
    predictions = model(input_values).ravel().tolist()
assert all(math.isfinite(value) for value in predictions)
print({"male_branch": predictions[0], "female_branch": predictions[1]})

weights_dir = Path("../weights")
weights_dir.mkdir(parents=True, exist_ok=True)
torch.save(model, weights_dir / f"{model.metadata['clock_name']}.pt")



%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'McGreevy, K. M., et al. “DNAmFitAge: biological age indicator '
             'incorporating physical fitness.” Aging 15(10): 3904–3938 (2023).',
 'citations': 99,
 'citations_date': '2026-07-05',
 'clock_name': 'dnamfitagegrip',
 'data_type': 'DNA methylation',
 'doi': 'https://doi.org/10.18632/aging.204538',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'model_type': 'LASSO regression',
 'n_features': 183,
 'notes': 'Sex-gated blood DNAm maximum-handgrip-strength estimator using the '
          'published female and male regressions selected by the female input.',
 'platform': ['Illumina 450K'],
 'population': 'adults',
 'predicts': ['grip strength'],
 'research_only': None,
 'species': 'Homo sapiens',
 'tissue': ['whole blood'],
 'training_target': ['grip strength'],
 'unit': ['kilograms'],
 'version'

## Clear temporary conversion inputs


In [6]:
# ruff: noqa: E501
for path in [repository, r_script_path, Path("FemaleMedians.csv"), Path("MaleMedians.csv"), Path("Grip_noAge_Females.csv"), Path("Grip_noAge_Males.csv")]:
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
